In [2]:
"""
Vacuum Cleaner World Simulator — AIMA Chapter 2
================================================
Modular implementation: sensors, actuators, environment, and agents
are all easily swappable/extendable.
"""

import random
from abc import ABC, abstractmethod


# ─────────────────────────────────────────────
#  ENVIRONMENT
# ─────────────────────────────────────────────

class VacuumEnvironment:
    """
    Represents the vacuum world.
    Configurable: size, initial dirt, and whether dirt can reappear.
    """

    def __init__(self, locations=None, dirt_prob=0.5, dynamic_dirt=False, dirty_rate=0.05):
        """
        Args:
            locations   : list of location names, e.g. ['A', 'B'] or ['A','B','C','D']
            dirt_prob   : probability each location starts dirty
            dynamic_dirt: whether clean squares can become dirty over time
            dirty_rate  : probability a clean square becomes dirty each step (if dynamic)
        """
        self.locations = locations or ['A', 'B']
        self.dynamic_dirt = dynamic_dirt
        self.dirty_rate = dirty_rate

        # State: dirt status per location
        self.dirt = {loc: (random.random() < dirt_prob) for loc in self.locations}

        # Agent starts at a random location
        self.agent_location = random.choice(self.locations)

        self.time = 0
        self.performance = 0

    def get_percept(self):
        """Sensor: returns (location, dirt_status)."""
        status = 'Dirty' if self.dirt[self.agent_location] else 'Clean'
        return (self.agent_location, status)

    def execute_action(self, action):
        """Actuator: apply the agent's chosen action."""
        loc = self.agent_location

        if action == 'Suck':
            if self.dirt[loc]:
                self.dirt[loc] = False

        elif action == 'Right':
            idx = self.locations.index(loc)
            if idx < len(self.locations) - 1:
                self.agent_location = self.locations[idx + 1]

        elif action == 'Left':
            idx = self.locations.index(loc)
            if idx > 0:
                self.agent_location = self.locations[idx - 1]

        elif action == 'NoOp':
            pass  # Do nothing

        else:
            raise ValueError(f"Unknown action: {action}")

    def update_performance(self, action):
        """
        Performance measure:
        +1 for each clean square at this time step
        -1 for each movement (optional cost)
        """
        clean_count = sum(1 for d in self.dirt.values() if not d)
        self.performance += clean_count
        if action in ('Left', 'Right'):
            self.performance -= 1  # movement cost

    def dynamic_update(self):
        """If dynamic_dirt is on, randomly dirty clean squares."""
        if self.dynamic_dirt:
            for loc in self.locations:
                if not self.dirt[loc] and random.random() < self.dirty_rate:
                    self.dirt[loc] = True

    def step(self, agent):
        """Run one simulation step."""
        percept = self.get_percept()
        action = agent.act(percept)
        self.execute_action(action)
        self.update_performance(action)
        self.dynamic_update()
        self.time += 1
        return percept, action

    def run(self, agent, steps=20, verbose=True):
        """Run the simulation for a given number of steps."""
        if verbose:
            print(f"\n{'='*55}")
            print(f"  VACUUM WORLD SIMULATOR")
            print(f"  Locations : {self.locations}")
            print(f"  Initial dirt: {self.dirt}")
            print(f"  Dynamic dirt: {self.dynamic_dirt}")
            print(f"  Agent type: {agent.__class__.__name__}")
            print(f"{'='*55}")
            print(f"{'Step':<6} {'Location':<10} {'Percept':<20} {'Action':<12} {'Score'}")
            print(f"{'-'*55}")

        for _ in range(steps):
            percept, action = self.step(agent)
            if verbose:
                print(f"{self.time:<6} {self.agent_location:<10} {str(percept):<20} {action:<12} {self.performance}")

        if verbose:
            print(f"\n  Final performance score: {self.performance}")
            print(f"  Final dirt state: {self.dirt}")
            print(f"{'='*55}\n")

        return self.performance


# ─────────────────────────────────────────────
#  AGENT BASE CLASS
# ─────────────────────────────────────────────

class Agent(ABC):
    """Abstract base class for all agents."""

    @abstractmethod
    def act(self, percept) -> str:
        """Given a percept, return an action."""
        pass


# ─────────────────────────────────────────────
#  AGENT IMPLEMENTATIONS
# ─────────────────────────────────────────────

class SimpleReflexAgent(Agent):
    """
    Simple reflex agent: acts only on current percept.
    If dirty → Suck. If in A → Right. If in B → Left.
    """

    def act(self, percept):
        location, status = percept
        if status == 'Dirty':
            return 'Suck'
        elif location == 'A':
            return 'Right'
        else:
            return 'Left'


class ModelBasedAgent(Agent):
    """
    Model-based agent: tracks which squares have been cleaned.
    Stops moving (NoOp) once it knows all squares are clean.
    Requires internal state → handles movement cost rationally.
    """

    def __init__(self, locations):
        self.locations = locations
        self.clean_memory = {loc: False for loc in locations}  # internal model

    def act(self, percept):
        location, status = percept

        if status == 'Dirty':
            return 'Suck'

        # Update internal model
        self.clean_memory[location] = True

        # If all known clean → NoOp (avoid movement cost)
        if all(self.clean_memory.values()):
            return 'NoOp'

        # Otherwise explore: move to next uncleaned location
        idx = self.locations.index(location)
        for i, loc in enumerate(self.locations):
            if not self.clean_memory[loc]:
                return 'Right' if i > idx else 'Left'

        return 'NoOp'


class RandomAgent(Agent):
    """Selects actions uniformly at random — baseline comparison."""

    ACTIONS = ['Left', 'Right', 'Suck', 'NoOp']

    def act(self, percept):
        return random.choice(self.ACTIONS)


# ─────────────────────────────────────────────
#  MAIN — Run & Compare Agents
# ─────────────────────────────────────────────

def compare_agents(steps=50, trials=100, locations=None):
    """
    Run multiple trials and report average performance per agent type.
    """
    locations = locations or ['A', 'B']
    agents = {
        'SimpleReflexAgent': lambda: SimpleReflexAgent(),
        'ModelBasedAgent'  : lambda: ModelBasedAgent(locations),
        'RandomAgent'      : lambda: RandomAgent(),
    }

    print(f"\n{'='*45}")
    print(f"  COMPARISON: {trials} trials × {steps} steps")
    print(f"  Locations: {locations}")
    print(f"{'='*45}")

    for name, agent_factory in agents.items():
        scores = []
        for _ in range(trials):
            env = VacuumEnvironment(locations=locations, dirt_prob=0.5)
            score = env.run(agent_factory(), steps=steps, verbose=False)
            scores.append(score)
        avg = sum(scores) / len(scores)
        print(f"  {name:<22} avg score: {avg:.1f}")

    print(f"{'='*45}\n")


if __name__ == '__main__':

    # ── Single verbose run of simple reflex agent ──
    env = VacuumEnvironment(locations=['A', 'B'], dirt_prob=0.7)
    agent = SimpleReflexAgent()
    env.run(agent, steps=10)

    # ── Single verbose run of model-based agent ──
    env2 = VacuumEnvironment(locations=['A', 'B', 'C'], dirt_prob=0.7)
    agent2 = ModelBasedAgent(['A', 'B', 'C'])
    env2.run(agent2, steps=15)

    # ── Compare all agents over many trials ──
    compare_agents(steps=50, trials=200, locations=['A', 'B'])


  VACUUM WORLD SIMULATOR
  Locations : ['A', 'B']
  Initial dirt: {'A': True, 'B': True}
  Dynamic dirt: False
  Agent type: SimpleReflexAgent
Step   Location   Percept              Action       Score
-------------------------------------------------------
1      B          ('B', 'Dirty')       Suck         1
2      A          ('B', 'Clean')       Left         1
3      A          ('A', 'Dirty')       Suck         3
4      B          ('A', 'Clean')       Right        4
5      A          ('B', 'Clean')       Left         5
6      B          ('A', 'Clean')       Right        6
7      A          ('B', 'Clean')       Left         7
8      B          ('A', 'Clean')       Right        8
9      A          ('B', 'Clean')       Left         9
10     B          ('A', 'Clean')       Right        10

  Final performance score: 10
  Final dirt state: {'A': False, 'B': False}


  VACUUM WORLD SIMULATOR
  Locations : ['A', 'B', 'C']
  Initial dirt: {'A': False, 'B': True, 'C': False}
  Dynamic dirt: 

In [3]:
"""
AIMA Exercise 16 — Stochastic Vacuum World Simulator
=====================================================

Extension of the above code to also include stochastic variants of the vacuum world, as described in AIMA Chapter 2 Exercise 16.


Simulates two stochastic variants:
  1. Murphy's Law  : Suck fails 25% of time; sensor wrong 10% of time
  2. Small Children: Each clean square has 10% chance of becoming dirty each step

Compares multiple agent strategies under each scenario.
"""

# ══════════════════════════════════════════════════════════
#  BASE ENVIRONMENT
# ══════════════════════════════════════════════════════════

class VacuumEnvironment(ABC):
    def __init__(self, locations=None, dirt_prob=0.5):
        self.locations   = locations or ['A', 'B']
        self.dirt        = {loc: (random.random() < dirt_prob) for loc in self.locations}
        self.agent_loc   = random.choice(self.locations)
        self.time        = 0
        self.performance = 0

    def _true_percept(self):
        return (self.agent_loc, 'Dirty' if self.dirt[self.agent_loc] else 'Clean')

    def get_percept(self):
        """Override in subclasses to add sensor noise."""
        return self._true_percept()

    def _move(self, direction):
        idx = self.locations.index(self.agent_loc)
        if direction == 'Right' and idx < len(self.locations) - 1:
            self.agent_loc = self.locations[idx + 1]
        elif direction == 'Left' and idx > 0:
            self.agent_loc = self.locations[idx - 1]

    def execute_action(self, action):
        """Override in subclasses to add action noise."""
        loc = self.agent_loc
        if action == 'Suck':
            self.dirt[loc] = False
        elif action in ('Left', 'Right'):
            self._move(action)

    def update_performance(self, action):
        self.performance += sum(1 for d in self.dirt.values() if not d)
        if action in ('Left', 'Right'):
            self.performance -= 1  # movement cost

    def post_step_update(self):
        """Override for dynamic environments."""
        pass

    def step(self, agent):
        percept = self.get_percept()
        action  = agent.act(percept)
        self.execute_action(action)
        self.update_performance(action)
        self.post_step_update()
        self.time += 1
        return percept, action

    def run(self, agent, steps=30, verbose=True):
        if verbose:
            print(f"\n{'═'*62}")
            print(f"  {self.__class__.__name__} | Agent: {agent.__class__.__name__}")
            print(f"  Locations: {self.locations}  |  Initial dirt: {self.dirt}")
            print(f"{'═'*62}")
            print(f"{'t':<5} {'Loc':<5} {'Percept':<22} {'Action':<14} {'Score'}")
            print(f"{'─'*62}")

        for _ in range(steps):
            percept, action = self.step(agent)
            if verbose:
                print(f"{self.time:<5} {self.agent_loc:<5} {str(percept):<22} {action:<14} {self.performance}")

        if verbose:
            print(f"\n  ✔ Final score : {self.performance}")
            print(f"  ✔ Dirt state  : {self.dirt}")
            print(f"{'═'*62}\n")

        return self.performance


# ══════════════════════════════════════════════════════════
#  SCENARIO 1 — MURPHY'S LAW
# ══════════════════════════════════════════════════════════

class MurphysLawEnvironment(VacuumEnvironment):
    """
    Suck action:
      - If dirty  : 75% cleans it, 25% does nothing
      - If clean  : 75% stays clean, 25% DEPOSITS dirt
    Sensor:
      - 10% chance of returning wrong status
    """

    SUCK_FAIL_PROB   = 0.25
    SENSOR_FAIL_PROB = 0.10

    def get_percept(self):
        loc, true_status = self._true_percept()
        # Sensor noise
        if random.random() < self.SENSOR_FAIL_PROB:
            noisy_status = 'Clean' if true_status == 'Dirty' else 'Dirty'
            return (loc, noisy_status)
        return (loc, true_status)

    def execute_action(self, action):
        loc = self.agent_loc
        if action == 'Suck':
            if random.random() < self.SUCK_FAIL_PROB:
                # Murphy strikes: flip dirt state
                self.dirt[loc] = not self.dirt[loc]
            else:
                self.dirt[loc] = False          # success: clean it
        elif action in ('Left', 'Right'):
            self._move(action)


# ══════════════════════════════════════════════════════════
#  SCENARIO 2 — SMALL CHILDREN
# ══════════════════════════════════════════════════════════

class SmallChildrenEnvironment(VacuumEnvironment):
    """
    After each step, every clean square has a 10% chance of becoming dirty.
    No action/sensor noise.
    """

    DIRT_RATE = 0.10

    def post_step_update(self):
        for loc in self.locations:
            if not self.dirt[loc] and random.random() < self.DIRT_RATE:
                self.dirt[loc] = True


# ══════════════════════════════════════════════════════════
#  AGENTS
# ══════════════════════════════════════════════════════════

class Agent(ABC):
    @abstractmethod
    def act(self, percept) -> str:
        pass


class SimpleReflexAgent(Agent):
    """Baseline: suck if dirty, else bounce between A and B."""
    def act(self, percept):
        loc, status = percept
        if status == 'Dirty':
            return 'Suck'
        return 'Right' if loc == 'A' else 'Left'


# ── Murphy's Law Agents ────────────────────────────────────

class MurphyReflexAgent(Agent):
    """
    Handles Murphy's Law by re-verifying after each Suck.
    Uses a retry counter to avoid infinite loops on bad luck.
    """
    MAX_RETRIES = 4

    def __init__(self):
        self.last_action  = None
        self.retry_count  = 0

    def act(self, percept):
        loc, status = percept

        # If we just sucked and it's still dirty → retry (up to MAX_RETRIES)
        if self.last_action == 'Suck' and status == 'Dirty':
            if self.retry_count < self.MAX_RETRIES:
                self.retry_count += 1
                self.last_action = 'Suck'
                return 'Suck'
            else:
                self.retry_count = 0  # give up, move on

        if status == 'Dirty':
            self.retry_count = 0
            self.last_action = 'Suck'
            return 'Suck'

        self.last_action = 'Right' if loc == 'A' else 'Left'
        return self.last_action


class MurphyModelAgent(Agent):
    """
    Full model-based agent for Murphy's Law + sensor noise.
    Maintains a belief state P(dirty) per location using Bayesian updating.
    Acts based on probability thresholds.
    """
    SUCK_FAIL   = 0.25
    SENSOR_FAIL = 0.10
    SUCK_THRESH = 0.30   # suck if P(dirty) > this
    MOVE_THRESH = 0.15   # move on if P(dirty) < this after retries

    def __init__(self, locations):
        self.locations = locations
        # Start with prior P(dirty) = 0.5 for each square
        self.belief = {loc: 0.5 for loc in locations}
        self.last_action = None

    def _bayesian_update(self, loc, percept_status):
        """Update P(dirty | percept) using Bayes' rule."""
        p = self.belief[loc]
        sf = self.SENSOR_FAIL

        if percept_status == 'Dirty':
            # P(percept=Dirty | dirty)*P(dirty) / P(percept=Dirty)
            likelihood_dirty  = (1 - sf)
            likelihood_clean  = sf
        else:
            likelihood_dirty  = sf
            likelihood_clean  = (1 - sf)

        numerator = likelihood_dirty * p
        denom     = numerator + likelihood_clean * (1 - p)
        self.belief[loc] = numerator / denom if denom > 0 else p

    def _update_after_suck(self, loc):
        """After sucking, update belief accounting for Murphy's Law."""
        p = self.belief[loc]
        # P(still dirty after suck) = P(dirty)*fail + P(clean)*fail (deposit)
        fail = self.SUCK_FAIL
        self.belief[loc] = p * fail + (1 - p) * fail

    def act(self, percept):
        loc, status = percept
        self._bayesian_update(loc, status)

        if self.last_action == 'Suck':
            self._update_after_suck(loc)

        if self.belief[loc] > self.SUCK_THRESH:
            self.last_action = 'Suck'
            return 'Suck'

        # Move to location with highest belief of being dirty
        other_locs = [l for l in self.locations if l != loc]
        if other_locs:
            target = max(other_locs, key=lambda l: self.belief[l])
            idx_now = self.locations.index(loc)
            idx_tgt = self.locations.index(target)
            action = 'Right' if idx_tgt > idx_now else 'Left'
        else:
            action = 'NoOp'

        self.last_action = action
        return action


# ── Small Children Agents ──────────────────────────────────

class PatrolAgent(Agent):
    """
    Rational agent for dynamic dirt (small children).
    Continuously patrols all locations in order — never stops.
    Prioritizes sucking over moving.
    """
    def __init__(self, locations):
        self.locations = locations
        self.direction = 1   # 1 = right, -1 = left

    def act(self, percept):
        loc, status = percept
        if status == 'Dirty':
            return 'Suck'
        # Bounce patrol
        idx = self.locations.index(loc)
        if idx >= len(self.locations) - 1:
            self.direction = -1
        elif idx <= 0:
            self.direction = 1
        return 'Right' if self.direction == 1 else 'Left'


class PriorityPatrolAgent(Agent):
    """
    Smarter patrol: tracks time since each square was last cleaned.
    Moves toward the square that has been clean the longest
    (most likely to be dirty again given 10% rate).
    """
    def __init__(self, locations):
        self.locations  = locations
        self.last_clean = {loc: 0 for loc in locations}
        self.time       = 0

    def act(self, percept):
        loc, status = percept
        self.time += 1

        if status == 'Dirty':
            self.last_clean[loc] = self.time
            return 'Suck'

        self.last_clean[loc] = self.time

        # Find location that has been clean the longest
        target = max(
            [l for l in self.locations if l != loc],
            key=lambda l: self.time - self.last_clean[l],
            default=loc
        )

        if target == loc:
            return 'NoOp'

        idx_now = self.locations.index(loc)
        idx_tgt = self.locations.index(target)
        return 'Right' if idx_tgt > idx_now else 'Left'


# ══════════════════════════════════════════════════════════
#  COMPARISON RUNNER
# ══════════════════════════════════════════════════════════

def compare(scenario_name, env_factory, agent_configs, steps=50, trials=300):
    print(f"\n{'╔' + '═'*58 + '╗'}")
    print(f"  📊 SCENARIO: {scenario_name}")
    print(f"  {trials} trials × {steps} steps each")
    print(f"{'╚' + '═'*58 + '╝'}")
    print(f"  {'Agent':<28} {'Avg Score':>10}  {'Min':>8}  {'Max':>8}")
    print(f"  {'─'*28}  {'─'*8}  {'─'*8}  {'─'*8}")

    results = {}
    for name, agent_factory in agent_configs:
        scores = []
        for _ in range(trials):
            env   = env_factory()
            score = env.run(agent_factory(), steps=steps, verbose=False)
            scores.append(score)
        avg = sum(scores) / len(scores)
        results[name] = avg
        print(f"  {name:<28} {avg:>10.1f}  {min(scores):>8}  {max(scores):>8}")

    winner = max(results, key=results.get)
    print(f"\n  🏆 Best agent: {winner} ({results[winner]:.1f})\n")


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════

if __name__ == '__main__':

    LOCS = ['A', 'B']

    # ── Verbose single runs ────────────────────────────────

    print("\n" + "▓"*62)
    print("  SCENARIO 1: MURPHY'S LAW  (single verbose run)")
    print("▓"*62)
    MurphysLawEnvironment(LOCS).run(MurphyModelAgent(LOCS), steps=12)

    print("\n" + "▓"*62)
    print("  SCENARIO 2: SMALL CHILDREN  (single verbose run)")
    print("▓"*62)
    SmallChildrenEnvironment(LOCS).run(PriorityPatrolAgent(LOCS), steps=12)

    # ── Statistical comparison ─────────────────────────────

    compare(
        scenario_name = "MURPHY'S LAW",
        env_factory   = lambda: MurphysLawEnvironment(LOCS, dirt_prob=0.5),
        agent_configs = [
            ("SimpleReflexAgent",  lambda: SimpleReflexAgent()),
            ("MurphyReflexAgent",  lambda: MurphyReflexAgent()),
            ("MurphyModelAgent",   lambda: MurphyModelAgent(LOCS)),
        ],
        steps=50, trials=400
    )

    compare(
        scenario_name = "SMALL CHILDREN",
        env_factory   = lambda: SmallChildrenEnvironment(LOCS, dirt_prob=0.5),
        agent_configs = [
            ("SimpleReflexAgent",  lambda: SimpleReflexAgent()),
            ("PatrolAgent",        lambda: PatrolAgent(LOCS)),
            ("PriorityPatrolAgent",lambda: PriorityPatrolAgent(LOCS)),
        ],
        steps=50, trials=400
    )


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  SCENARIO 1: MURPHY'S LAW  (single verbose run)
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

══════════════════════════════════════════════════════════════
  MurphysLawEnvironment | Agent: MurphyModelAgent
  Locations: ['A', 'B']  |  Initial dirt: {'A': True, 'B': False}
══════════════════════════════════════════════════════════════
t     Loc   Percept                Action         Score
──────────────────────────────────────────────────────────────
1     A     ('B', 'Clean')         Left           0
2     A     ('A', 'Dirty')         Suck           2
3     B     ('A', 'Clean')         Right          3
4     A     ('B', 'Clean')         Left           4
5     A     ('A', 'Dirty')         Suck           6
6     B     ('A', 'Clean')         Right          7
7     A     ('B', 'Clean')         Left           8
8     A     ('A', 'Dirty')         Suck           10
9     B     ('A', 'Clean')         Right     